In [1]:
# --- TRANSFORMER TUNING & JSON SAVING (Silchar, Shillong, Lengpui, Imphal) ---

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import optuna
import json
import os
import gc

# --- 1. CONFIGURATION ---
LOCATIONS = ['silchar', 'shillong', 'lengpui', 'imphal']
N_TRIALS = 30  # 30 trials per location is a good balance
LOOK_BACK = 7
HORIZON = 3
GLOBAL_SEED = 42

# --- 2. DATA LOADING & PROCESSING ---
def load_and_process_data(location):
    # Try multiple paths for Kaggle/Local compatibility
    paths = [
        f'/kaggle/input/dataset2/{location}.csv', 
        f'{location}.csv', 
        f'/kaggle/input/dataset2/{location.lower()}.csv', 
        f'{location.lower()}.csv'
    ]
    df = None
    for p in paths:
        try: 
            df = pd.read_csv(p)
            print(f"  ✅ Data loaded from: {p}")
            break
        except: continue
        
    if df is None: return None, None, None

    # Preprocessing
    df['Date'] = pd.to_datetime({'year':df['YEAR'], 'month':df['MN'], 'day':df['DT']})
    df.set_index('Date', inplace=True)
    cols = ['RF','MSLP','DBT','RH','ONI','DMI']
    for c in cols: df[c] = pd.to_numeric(df[c], errors='coerce')
    daily = df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in cols})
    daily.index = pd.to_datetime(daily.index); daily.dropna(subset=cols, inplace=True)
    
    # Feature Engineering
    daily['RF_SSA'] = daily['RF'].rolling(7,center=True).mean().fillna(daily['RF'])
    daily['RF_7m'] = daily['RF'].rolling(7).mean().shift(1)
    daily['RF_7s'] = daily['RF'].rolling(7).std().shift(1)
    daily['sin_d'] = np.sin(2*np.pi*daily.index.dayofyear/365.25)
    daily['cos_d'] = np.cos(2*np.pi*daily.index.dayofyear/365.25)
    daily['RF_l1'] = daily['RF'].shift(1); daily['DBT_l1'] = daily['DBT'].shift(1)
    daily['RH_MSLP_I'] = (daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan)
    daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True)
    
    # Create Sequences
    feats = ['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']
    feats = [f for f in feats if f in daily.columns]
    
    X, Y = [], []
    ts = pd.to_numeric(daily['RF'].shift(-HORIZON), errors='coerce') 
    vfd = daily[feats]
    
    for i in range(len(vfd) - LOOK_BACK - HORIZON + 1):
        if not pd.isna(ts.iloc[i+LOOK_BACK-1]):
            X.append(vfd.iloc[i:i+LOOK_BACK].values)
            Y.append(ts.iloc[i+LOOK_BACK-1])
            
    X = np.array(X).astype(np.float32)
    Y = np.log1p(np.array(Y).astype(np.float32)) # Log Transform Targets
    
    return X, Y, len(feats)

# --- 3. TRANSFORMER MODEL ---
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = Dropout(dropout)(x); x = LayerNormalization(epsilon=1e-6)(x)
    res = Add()([x, inputs])
    x = Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = Dropout(dropout)(x); x = Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    return Add()([x, res])

# --- 4. OPTUNA OBJECTIVE ---
def create_objective(X_tr, Y_tr, X_va, Y_va, n_feats):
    Y_tr_p = (np.expm1(Y_tr) > 0).astype(int)
    Y_va_p = (np.expm1(Y_va) > 0).astype(int)
    
    def objective(trial):
        tf.keras.backend.clear_session()
        
        # Search Space
        head_size = trial.suggest_categorical('head_size', [32, 64, 128])
        num_heads = trial.suggest_categorical('num_heads', [2, 4])
        ff_dim = trial.suggest_int('ff_dim', 64, 256)
        num_blocks = trial.suggest_int('num_transformer_blocks', 1, 2) # 1-2 is usually enough for small data
        dropout = trial.suggest_float('dropout_rate', 0.1, 0.4)
        lr = trial.suggest_float('learning_rate', 1e-4, 5e-3, log=True)
        l2_v = trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
        mlp = trial.suggest_categorical('mlp_units', [64, 128])
        
        # Build Model
        inp = Input(shape=(LOOK_BACK, n_feats))
        x = inp
        for _ in range(num_blocks):
            x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
        
        x = GlobalAveragePooling1D()(x)
        x = Dense(mlp, activation="relu", kernel_regularizer=l2(l2_v))(x)
        x = Dropout(dropout)(x)
        
        p = Dense(1, activation='sigmoid', name='prob_output')(x)
        r = Dense(1, activation='linear', name='reg_output')(x)
        
        model = Model(inp, [p, r])
        model.compile(optimizer=Adam(learning_rate=lr),
                      loss={'prob_output': BinaryCrossentropy(), 'reg_output': 'mse'},
                      loss_weights={'prob_output': 0.2, 'reg_output': 1.8})
        
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        h = model.fit(X_tr, {'prob_output': Y_tr_p, 'reg_output': Y_tr},
                      validation_data=(X_va, {'prob_output': Y_va_p, 'reg_output': Y_va}),
                      epochs=50, batch_size=64, callbacks=[es], verbose=0)
        
        return min(h.history['val_loss'])
    return objective

# --- 5. MAIN LOOP & JSON SAVING ---
print(f"{'='*50}\n STARTING TUNING FOR: {LOCATIONS}\n{'='*50}")

for loc in LOCATIONS:
    print(f"\n>>> 📍 Processing: {loc.upper()}")
    
    # 1. Load Data
    X, Y, n_feats = load_and_process_data(loc)
    if X is None: 
        print(f"❌ Data not found for {loc}"); continue
        
    # 2. Split Data (80/20 for tuning)
    sp = int(len(X) * 0.8)
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[:sp].reshape(sp, -1)).reshape(sp, LOOK_BACK, n_feats)
    X_va = sc.transform(X[sp:].reshape(len(X)-sp, -1)).reshape(len(X)-sp, LOOK_BACK, n_feats)
    Y_tr, Y_va = Y[:sp], Y[sp:]
    
    # 3. Optimize
    study = optuna.create_study(direction='minimize')
    study.optimize(create_objective(X_tr, Y_tr, X_va, Y_va, n_feats), n_trials=N_TRIALS)
    
    # 4. Prepare Params for JSON
    best_params = study.best_params
    # Ensure mlp_units is a list (compatible with main model code)
    if not isinstance(best_params.get('mlp_units'), list):
        best_params['mlp_units'] = [best_params['mlp_units']]
        
    # 5. Save JSON immediately
    filename = f"HP_TRANS_{loc}.json"
    with open(filename, 'w') as f:
        json.dump(best_params, f, indent=4)
        
    print(f"✅ Best Params Found & Saved to '{filename}':")
    print(best_params)
    
    # 6. Cleanup
    gc.collect()

print("\n" + "="*50)
print("🎉 ALL JSON FILES SAVED SUCCESSFULLY")
print("="*50)

2025-11-22 20:42:20.857689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763844141.119097      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763844141.190608      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

✅ Best Params Found & Saved to 'HP_TRANS_shillong.json':
{'head_size': 128, 'num_heads': 4, 'ff_dim': 75, 'num_transformer_blocks': 2, 'dropout_rate': 0.21677138556844955, 'learning_rate': 0.0020209898883568264, 'l2_reg': 0.000718795160936171, 'mlp_units': [64]}

>>> 📍 Processing: LENGPUI
  ✅ Data loaded from: /kaggle/input/dataset2/lengpui.csv


[I 2025-11-22 21:31:38,218] A new study created in memory with name: no-name-cddebf7a-3c99-4d87-9dca-23c5cf9665c2
[I 2025-11-22 21:32:14,291] Trial 0 finished with value: 1.881341576576233 and parameters: {'head_size': 64, 'num_heads': 4, 'ff_dim': 209, 'num_transformer_blocks': 1, 'dropout_rate': 0.3172061967072556, 'learning_rate': 0.0006870311461224425, 'l2_reg': 2.0926942006871923e-05, 'mlp_units': 128}. Best is trial 0 with value: 1.881341576576233.
[I 2025-11-22 21:32:37,141] Trial 1 finished with value: 2.245290756225586 and parameters: {'head_size': 128, 'num_heads': 4, 'ff_dim': 244, 'num_transformer_blocks': 1, 'dropout_rate': 0.2360371246634819, 'learning_rate': 0.00028112586186142637, 'l2_reg': 0.00023618381745724244, 'mlp_units': 128}. Best is trial 0 with value: 1.881341576576233.
[I 2025-11-22 21:33:12,804] Trial 2 finished with value: 1.8569415807724 and parameters: {'head_size': 64, 'num_heads': 2, 'ff_dim': 192, 'num_transformer_blocks': 1, 'dropout_rate': 0.176813303

✅ Best Params Found & Saved to 'HP_TRANS_lengpui.json':
{'head_size': 32, 'num_heads': 4, 'ff_dim': 167, 'num_transformer_blocks': 2, 'dropout_rate': 0.1110748717931713, 'learning_rate': 0.004746180675989971, 'l2_reg': 3.905140291474256e-05, 'mlp_units': [128]}

>>> 📍 Processing: IMPHAL
  ✅ Data loaded from: /kaggle/input/dataset2/imphal.csv


[I 2025-11-22 21:53:40,961] A new study created in memory with name: no-name-def706f7-89e8-4931-89e0-b2e1b8d97cbe
[I 2025-11-22 21:54:25,358] Trial 0 finished with value: 1.1488306522369385 and parameters: {'head_size': 128, 'num_heads': 2, 'ff_dim': 126, 'num_transformer_blocks': 1, 'dropout_rate': 0.19837354914993685, 'learning_rate': 0.0047810831863050705, 'l2_reg': 0.00033626227876368205, 'mlp_units': 128}. Best is trial 0 with value: 1.1488306522369385.
[I 2025-11-22 21:55:09,759] Trial 1 finished with value: 1.5973076820373535 and parameters: {'head_size': 32, 'num_heads': 2, 'ff_dim': 243, 'num_transformer_blocks': 1, 'dropout_rate': 0.3361160263773848, 'learning_rate': 0.0006382510741530036, 'l2_reg': 4.702814903759909e-05, 'mlp_units': 128}. Best is trial 0 with value: 1.1488306522369385.
[I 2025-11-22 21:56:10,160] Trial 2 finished with value: 1.0054315328598022 and parameters: {'head_size': 64, 'num_heads': 4, 'ff_dim': 243, 'num_transformer_blocks': 2, 'dropout_rate': 0.114

✅ Best Params Found & Saved to 'HP_TRANS_imphal.json':
{'head_size': 64, 'num_heads': 4, 'ff_dim': 132, 'num_transformer_blocks': 2, 'dropout_rate': 0.1233459147756123, 'learning_rate': 0.003231465767523693, 'l2_reg': 0.0001305652983362326, 'mlp_units': [128]}

🎉 ALL JSON FILES SAVED SUCCESSFULLY
